# KWISMO — Notebook 01 : Analyse Exploratoire & Visualisations Avancées (EDA Global)

Ce notebook propose une **exploration graphique et statistique approfondie** pour les deux modèles de la suite KWISMO :
- **PARTIE 1 — Modèle A (Scoring Comportemental & Temporel)** : Analyse des distributions de risque, ratios de paresse, vélocités, déduplication d'appareils et corrélations des 11 caractéristiques.
- **PARTIE 2 — Modèle B (NLP, Argots & SMS Africains)** : Analyse des volumes par source, longueurs de textes, top 25 des n-grams, répartition des catégories de fraude et normalisation argotique.

---

In [ ]:
# 1. Détection de l'environnement, résolution de la racine du projet et injection de .venv dans sys.path
import os
import sys
import subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

current_dir = Path.cwd()
repo_root = current_dir
for candidate in [current_dir, current_dir.parent, current_dir.parent.parent]:
    if (candidate / "src").exists() and (candidate / "data").exists():
        repo_root = candidate.resolve()
        break
    elif (candidate / "kwismo-ai" / "src").exists():
        repo_root = (candidate / "kwismo-ai").resolve()
        break

os.chdir(repo_root)

# Injection prioritaire de l'environnement virtuel local .venv dans sys.path
local_venv_win = repo_root / ".venv" / "Scripts" / "python.exe"
local_venv_nix = repo_root / ".venv" / "bin" / "python"
if local_venv_win.exists():
    site_pkgs = repo_root / ".venv" / "Lib" / "site-packages"
    if site_pkgs.exists() and str(site_pkgs) not in sys.path:
        sys.path.insert(0, str(site_pkgs))
elif local_venv_nix.exists():
    site_pkgs = repo_root / ".venv" / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
    if site_pkgs.exists() and str(site_pkgs) not in sys.path:
        sys.path.insert(0, str(site_pkgs))

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Importation des modules une fois sys.path correctement configuré avec .venv
import json
import re
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration du thème graphique haut de gamme (Palette Crest & Mako)
sns.set_theme(style="whitegrid", palette="crest")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.titleweight"] = "bold"

print(f"✅ Environnement .venv résolu avec succès. Racine du projet : {repo_root}")

## 📊 PARTIE 1 : Visualisation Approfondie du Dataset Modèle A (Scoring Comportemental & Temporel)

Le jeu de données `data/processed/model_a_dataset.csv` regroupe 1200 numéros comportant leurs métriques de vérifications, vélocités, décroissance temporelle et gravité de fraude.

In [ ]:
model_a_path = repo_root / "data" / "processed" / "model_a_dataset.csv"

if not model_a_path.exists():
    print("⚙️ Dataset Modèle A introuvable. Génération automatique du dataset calibré...")
    try:
        from src.data.generate_model_a_data import generate_dataset
        df_a = generate_dataset(num_samples=1200)
    except Exception as err:
        print(f"⚠️ Erreur lors de la génération : {err}")
        df_a = None
else:
    df_a = pd.read_csv(model_a_path)

if df_a is not None:
    print(f"📌 Synthèse statistique Modèle A ({len(df_a)} lignes) :")
    display(df_a.describe().T.round(2))
    
    # --- PLANCHE DE GRAPHIQUES 1 : DISTRIBUTION & RELATIONS COMPORTEMENTALES ---
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    
    # Graphe 1.1 : Distribution Donut Pie Chart de la Classe de Risque
    label_counts = df_a["label"].value_counts()
    colors = ["#2A9D8F", "#E63946"]
    axes[0, 0].pie(label_counts, labels=["Sécurisé (0)", "Frauduleux (1)"], autopct="%1.1f%%", startangle=140, colors=colors, wedgeprops=dict(width=0.4, edgecolor='w'))
    axes[0, 0].set_title("1.1 Répartition de la Classe Cible (Equilibre Légitime vs Fraude)")
    
    # Graphe 1.2 : Boxplot comparatif des Vérifications par Classe
    sns.boxplot(data=df_a, x="label", y="nombre_verifications", palette=colors, ax=axes[0, 1])
    axes[0, 1].set_title("1.2 Nombre de Vérifications par Classe de Risque")
    axes[0, 1].set_xlabel("Classe de Risque (0: Sécurisé, 1: Fraude)")
    axes[0, 1].set_ylabel("Nombre de vérifications")
    
    # Graphe 1.3 : Scatterplot Vélocité des Vérifications vs Signalements
    sns.scatterplot(data=df_a, x="vitesse_verifications", y="vitesse_signalements", hue="label", style="label", palette=colors, ax=axes[1, 0], alpha=0.8)
    axes[1, 0].set_title("1.3 Vélocité des Vérifications vs Vélocité des Signalements (/jour)")
    axes[1, 0].set_xlabel("Vérifications par jour")
    axes[1, 0].set_ylabel("Signalements par jour")
    
    # Graphe 1.4 : Densité du Ratio de Paresse
    sns.kdeplot(data=df_a, x="ratio_verif_signalement", hue="label", fill=True, common_norm=False, palette=colors, ax=axes[1, 1])
    axes[1, 1].set_title("1.4 Densité du Ratio de Paresse (Verifications / Signalements)")
    axes[1, 1].set_xlabel("Ratio Verif / Signalement")
    
    plt.tight_layout()
    plt.show()
    
    # --- PLANCHE DE GRAPHIQUES 2 : CORRÉLATIONS & TIME DECAY ---
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Graphe 1.5 : Matrice de Corrélation Complète Heatmap
    num_cols = df_a.select_dtypes(include=[np.number]).columns
    corr = df_a[num_cols].corr()
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="mako", ax=axes[0], cbar=True, square=True)
    axes[0].set_title("1.5 Heatmap des Corrélations de Pearson (11 Caractéristiques)")
    
    # Graphe 1.6 : Signalements Bruts vs Signalements Effectifs Pondérés (Time Decay & Device Deduplication)
    sns.regplot(data=df_a, x="nombre_signalements", y="signalements_effectifs_ponderes", color="#457B9D", ax=axes[1], scatter_kws={"alpha": 0.6})
    axes[1].set_title("1.6 Effet de la Décroissance Temporelle & Anti-Vengeance")
    axes[1].set_xlabel("Nombre Brut de Signalements")
    axes[1].set_ylabel("Signalements Effectifs Pondérés")
    
    plt.tight_layout()
    plt.show()
else:
    print("⏩ Modèle A non analysable.")

## 📝 PARTIE 2 : Visualisation Approfondie du Dataset Modèle B (NLP, Multilinguisme & Dialectes)

Analyse statistique des textes issus de `model_b_augmented.jsonl`, `model_b_clean.jsonl`, `legit_examples.jsonl` et du dictionnaire argotique `slang_dictionary.json`.

In [ ]:
possible_b_paths = [
    repo_root / "data" / "processed" / "model_b_augmented.jsonl",
    repo_root / "data" / "processed" / "model_b_clean.jsonl",
    repo_root / "data" / "processed" / "legit_examples.jsonl",
    repo_root / "data" / "raw" / "scraped" / "messages.jsonl"
]

records_b = []
for p in possible_b_paths:
    if p.exists():
        try:
            with open(p, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        item = json.loads(line)
                        item["_source_file"] = p.name
                        records_b.append(item)
        except Exception:
            continue

if records_b:
    df_b = pd.DataFrame(records_b)
    print(f"📌 Synthèse volumétrique Modèle B ({len(df_b)} messages extraits de {len(possible_b_paths)} fichiers) :")
    
    text_col = None
    for col in ["description", "texte", "text", "content"]:
        if col in df_b.columns:
            text_col = col
            break
            
    if text_col:
        df_b["longueur_caracteres"] = df_b[text_col].fillna("").apply(len)
        df_b["nombre_mots"] = df_b[text_col].fillna("").apply(lambda t: len(t.split()))
        
        # --- PLANCHE DE GRAPHIQUES 3 : VOLUMÉTRIE & LONGUEURS DE TEXTES ---
        fig, axes = plt.subplots(2, 2, figsize=(15, 11))
        
        # Graphe 2.1 : Volume de Messages par Fichier Source
        source_counts = df_b["_source_file"].value_counts()
        sns.barplot(x=source_counts.values, y=source_counts.index, palette="viridis", ax=axes[0, 0])
        axes[0, 0].set_title("2.1 Répartition du Volume par Fichier de Données")
        axes[0, 0].set_xlabel("Nombre de messages")
        
        # Graphe 2.2 : Distribution du Nombre de Mots (KDE)
        sns.histplot(df_b["nombre_mots"], kde=True, color="#2A9D8F", bins=30, ax=axes[0, 1])
        axes[0, 1].set_title("2.2 Distribution de la Longueur des Messages (Mots)")
        axes[0, 1].set_xlabel("Nombre de mots")
        
        # Graphe 2.3 : Top 20 des Mots les Plus Fréquents (Hors Stopwords)
        stopwords = {"de", "la", "le", "les", "des", "un", "une", "et", "a", "en", "du", "pour", "sur", "est", "pas", "plus", "par", "que", "dans", "avec", "au", "ce", "qui", "ne", "http", "https", "com"}
        all_words = []
        for txt in df_b[text_col].dropna():
            words = re.findall(r"\w+", str(txt).lower())
            all_words.extend([w for w in words if w not in stopwords and len(w) > 2 and not w.isdigit()])
            
        top20 = Counter(all_words).most_common(20)
        top20_df = pd.DataFrame(top20, columns=["Mot", "Fréquence"])
        sns.barplot(data=top20_df, x="Fréquence", y="Mot", palette="Spectral", ax=axes[1, 0])
        axes[1, 0].set_title("2.3 Top 20 des Termes Répétés dans les Témoignages")
        
        # Graphe 2.4 : Autocatégorisation NLP par le Modèle B
        try:
            from src.models.model_b.preprocess import categorize_description
            df_b["categorie_ia"] = df_b[text_col].dropna().apply(categorize_description)
            cat_counts = df_b["categorie_ia"].value_counts().reset_index()
            cat_counts.columns = ["Catégorie", "Total"]
            sns.barplot(data=cat_counts, x="Total", y="Catégorie", palette="rocket", ax=axes[1, 1])
            axes[1, 1].set_title("2.4 Répartition des Catégories Détectées Autonomement par l'IA")
        except Exception as err:
            axes[1, 1].text(0.5, 0.5, f"Catégorisation non disponible : {err}", ha="center")
            
        plt.tight_layout()
        plt.show()
        
        # --- PLANCHE DE GRAPHIQUES 4 : ANALYSE DES DIALECTES & DICTIONNAIRE ARGOTIQUE ---
        slang_path = repo_root / "src" / "data" / "slang_dictionary.json"
        if slang_path.exists():
            try:
                slang_dict = json.loads(slang_path.read_text(encoding="utf-8")).get("terms", {})
                slang_counts = {}
                for term in slang_dict.keys():
                    pattern = re.compile(r"\b" + re.escape(term) + r"\b", re.IGNORECASE)
                    count = sum(1 for txt in df_b[text_col].dropna() if pattern.search(str(txt)))
                    if count > 0:
                        slang_counts[term] = count
                
                if slang_counts:
                    slang_df = pd.DataFrame(list(slang_counts.items()), columns=["Terme Argotique", "Occurrences"]).sort_values(by="Occurrences", ascending=False).head(15)
                    plt.figure(figsize=(12, 5))
                    sns.barplot(data=slang_df, x="Occurrences", y="Terme Argotique", palette="flare")
                    plt.title("2.5 Fréquence des Termes Argotiques Africains Détectés (Camfranglais, Pidgin, SMS)", fontsize=12, fontweight="bold")
                    plt.tight_layout()
                    plt.show()
            except Exception as err:
                print(f"⚠️ Erreur lors de l'analyse du dictionnaire argotique : {err}")
else:
    print("⏩ Modèle B non analysable.")